In [1]:
print("nfgnjg")

nfgnjg


In [1]:
%pwd
import os
os.chdir("../")
#Just went one step bcakword with this directory....

%pwd

'd:\\mlops\\Crypto_Guardian'

In [25]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path 
    model_path: Path
    vectorizer_path: Path
    test_embedding: Path
    evaluation_metrics_path : Path
    curve_img: Path
    mlflow_tracking_uri: str
    mlflow_experiment_name: str
    mlflow_registered_model_name :str
    all_parmas: dict

In [10]:
print("jsdnfgjnfg")

jsdnfgjnfg


In [26]:

from src.Crypto.constants import *
from src.Crypto.utils.helper import read_yaml,create_directories,save_json
from src.Crypto import logger

class ConfigurationManager:
    def __init__(self,config_file_path=CONFIG_FILE_PATH,
                      params_file_path = PARAMS_FILE_PATH,
                      schema_file_path = SCHEMA_FILE_PATH):
        self.config =read_yaml( config_file_path)
        self.parmas = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)
        
        # create_directories([self.config.artifacts_root])
        logger.debug(f"Till Now all the Yaml Files are Read Sucessfully...✅")
    def get_model_evaluation(self)->ModelEvaluationConfig:
        config = self.config.model_evaluation
        mlflow_config = self.config.mlflow
        parmas = self.parmas.SVC
        create_directories([config.root_dir])
        
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            test_embedding= config.test_embedding,
            vectorizer_path = config.vectorizer_path,
            mlflow_tracking_uri=mlflow_config.mlflow_tracking_uri,
            mlflow_experiment_name=mlflow_config.mlflow_experiment_name,
            mlflow_registered_model_name=mlflow_config.mlflow_registered_model_name,
            evaluation_metrics_path=config.evaluation_metrics_path,
            curve_img= config.curve_img,
            all_parmas = parmas
            # f1_score_threshold=config.f1_score_threshold
        )
        logger.debug("Model Evaluation is working compeletely fine...✅")
        return model_evaluation_config



In [46]:



import numpy as np
import pickle
import json
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from src.Crypto import logger
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, roc_auc_score
import os
from dotenv import load_dotenv
load_dotenv()

class ModelEvaluationComponent:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        self.x_test_emb = np.load(self.config.test_embedding)
        self.test_df = pd.read_csv(self.config.test_data_path)
        self.y_test = self.test_df['Label']
        self.vectorizer = joblib.load(self.config.vectorizer_path)
        self.model = joblib.load(self.config.model_path)
        
    def evaluate(self):
        self.y_pred = self.model.predict(self.x_test_emb)
        self.y_score = self.model.predict_proba(self.x_test_emb)[:, 1] 
        
        # Calculate metrics
        logger.info("Evaluation metrics calculate kar rahe hain...")
        accuracy = accuracy_score(self.y_test, self.y_pred)
        precision = precision_score(self.y_test, self.y_pred, average='weighted')
        recall = recall_score(self.y_test, self.y_pred, average='weighted')
        f1 = f1_score(self.y_test, self.y_pred, average='weighted')
        roc_auc = roc_auc_score(self.y_test, self.y_score)
        
        # Plots (ROC & PR)
        os.makedirs(self.config.root_dir, exist_ok=True)
        roc_plot_path = os.path.join(self.config.root_dir, "roc_curve.png")
        pr_plot_path = os.path.join(self.config.root_dir, "pr_curve.png")
        
        RocCurveDisplay.from_estimator(self.model, self.x_test_emb, self.y_test)
        plt.savefig(roc_plot_path)
        plt.close()

        PrecisionRecallDisplay.from_estimator(self.model, self.x_test_emb, self.y_test)
        plt.savefig(pr_plot_path)
        plt.close()

        # MLFLOW
        mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
        mlflow.set_experiment(self.config.mlflow_experiment_name)
        
        with mlflow.start_run() as run:
            mlflow.log_params(self.config.all_parmas)
            mlflow.log_param("vectorizer_model", "all-mpnet-base-v2")
            mlflow.log_metrics({"accuracy": accuracy, "precision": precision, "recall": recall, "f1_score": f1, "roc_auc": roc_auc})
            
            mlflow.log_artifact(roc_plot_path, "plots")
            mlflow.log_artifact(pr_plot_path, "plots")
            
            # Model Logging (Pipeline skip kar di taaki fast ho)
            model_info = mlflow.sklearn.log_model(
                sk_model=self.model,
                artifact_path="model",
                registered_model_name=self.config.mlflow_registered_model_name
            )

            current_version = model_info.registered_model_version
            logger.info(f"Model version {current_version} register ho gaya!")

            # FIX 1: Tag set karne ka sahi tareeka
            client = MlflowClient()
            client.set_model_version_tag(
                name=self.config.mlflow_registered_model_name,
                version=current_version,
                key="f1_score",
                value=str(f1)
            )

            # Promotion Logic call
            should_promote = self.check_and_promote_model(f1, current_version)
            
            if should_promote:
                logger.info(f"Version {current_version} PRODUCTION mein hai! 🚀")
            else:
                logger.info(f"Version {current_version} STAGING mein rakha gaya.")

    def check_and_promote_model(self, current_f1, current_version):
        try:
            client = MlflowClient()
            model_name = self.config.mlflow_registered_model_name
            
            # Latest production version check
            prod_versions = client.get_latest_versions(model_name, stages=['Production'])
            
            if prod_versions:
                prod_model = prod_versions[0]
                # Tag se F1 score nikalna (safety ke liye 0 default)
                prod_f1 = float(prod_model.tags.get("f1_score", 0))
                
                if current_f1 > prod_f1:
                    logger.info(f"New model (F1: {current_f1}) better hai old (F1: {prod_f1}) se.")
                    # Purane ko hatao
                    client.transition_model_version_stage(model_name, prod_model.version, "Archived")
                    # Naye ko promote karo
                    client.transition_model_version_stage(model_name, current_version, "Production")
                    return True
                else:
                    client.transition_model_version_stage(model_name, current_version, "Staging")
                    return False
            else:
                # Pehla production model
                client.transition_model_version_stage(model_name, current_version, "Production")
                return True
        except Exception as e:
            logger.error(f"Promotion error: {e}")
            return False
                        

In [47]:
try:
    cfm = ConfigurationManager()
    data_ingestion = cfm.get_model_evaluation()
    data_ingestion_component = ModelEvaluationComponent(data_ingestion)
    data_ingestion_component.evaluate()
    # data_ingestion_component.train_model()
    logger.info("Pipeline Ran Sucessfully...✅")
except Exception as e:
    logger.error("Pipeline Error...❌")
    raise e

[06-05-2026 07:40:29 PM - helper - INFO - Yaml File :config\config.yaml Read Sucessfully✅]
[06-05-2026 07:40:29 PM - helper - INFO - Yaml File :params.yaml Read Sucessfully✅]
[06-05-2026 07:40:29 PM - helper - INFO - Yaml File :schema.yaml Read Sucessfully✅]
[06-05-2026 07:40:29 PM - helper - INFO - Folder Created Sucessfully: artifacts/model_evaluation]
[06-05-2026 07:40:30 PM - 3334215264 - INFO - Evaluation metrics calculate kar rahe hain...]


2026/05/06 19:40:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 19:40:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'sentiment_model' already exists. Creating a new version of this model...
2026/05/06 19:41:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: sentiment_model, version 5
Created version '5' of model 'sentiment_model'.


[06-05-2026 07:41:12 PM - 3334215264 - INFO - Model version 5 register ho gaya!]


C:\Users\sahil\AppData\Local\Temp\ipykernel_14760\3334215264.py:97: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  prod_versions = client.get_latest_versions(model_name, stages=['Production'])


[06-05-2026 07:41:14 PM - 3334215264 - INFO - New model (F1: 0.7640971817298348) better hai old (F1: 0.0) se.]


C:\Users\sahil\AppData\Local\Temp\ipykernel_14760\3334215264.py:107: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(model_name, prod_model.version, "Archived")
C:\Users\sahil\AppData\Local\Temp\ipykernel_14760\3334215264.py:109: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(model_name, current_version, "Production")


[06-05-2026 07:41:15 PM - 3334215264 - INFO - Version 5 PRODUCTION mein hai! 🚀]
🏃 View run legendary-croc-773 at: https://dagshub.com/sahilkumarrock8084/Youtube-Sentiment-Analysis.mlflow/#/experiments/0/runs/d4c38f97695d4a4baa5e579fdebaf87f
🧪 View experiment at: https://dagshub.com/sahilkumarrock8084/Youtube-Sentiment-Analysis.mlflow/#/experiments/0
[06-05-2026 07:41:16 PM - 3250187576 - INFO - Pipeline Ran Sucessfully...✅]
